In [ ]:

# Bank Customer Churn Prediction using Neural Network

# 1. Import libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping


# 2. Read dataset
df = pd.read_csv("Churn_Modelling.csv")

print("First five records:")
print(df.head())


# 3. Separate features and target
X = df.drop(
    columns=['RowNumber', 'CustomerId', 'Surname', 'Exited']
)

y = df['Exited']


# Convert categorical data into numerical data
X = pd.get_dummies(
    X,
    columns=['Geography', 'Gender'],
    drop_first=True
)


# 4. Split dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


# 5. Normalize the data
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)

X_test = scaler.transform(X_test)


# 6. Build the neural network model
model = Sequential([
    Dense(32, activation='relu',
          input_shape=(X_train.shape[1],)),

    Dropout(0.2),

    Dense(16, activation='relu'),

    Dense(1, activation='sigmoid')
])


model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)


# 7. Train and improve the model using Early Stopping
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)


model.fit(
    X_train,
    y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)


# 8. Predict customer churn
y_prob = model.predict(X_test)

y_pred = (y_prob >= 0.5).astype(int).ravel()


# 9. Print accuracy and confusion matrix
print("\nAccuracy:", accuracy_score(y_test, y_pred))

print("\nConfusion Matrix:\n",
      confusion_matrix(y_test, y_pred))

print("\nClassification Report:\n",
      classification_report(y_test, y_pred))


# 10. Conclusion
print("\n0 = Customer stays, 1 = Customer leaves")

print("The neural network predicts whether a customer will leave the bank.")
